# 4 — High-gap posts + Embedding-based Mental-Health Flags

This notebook uses **cosine similarity** to a small set of **mental-health seed sentences** to create:

- `mh_sim_max`: maximum cosine similarity of a post to any seed sentence
- `mh_flag`: a binary flag based on either a **quantile** threshold or a **fixed** cutoff
- `mh_best_seed`: which seed sentence matched best (and the similarity score)

## Inputs
- `data/processed/3-gap_and_swear_features.csv` (from Notebook 3)

## Outputs
- `data/processed/writestreak_posts_highgap_with_mh_embedding.csv` (high-gap subset with mh signals)
- optional: `data/processed/writestreak_posts_all_with_mh_embedding.csv` (if "all posts" section enabled)


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
processed_dir = REPO_ROOT / ".." / "data" / "processed"

notebook3DataPath = processed_dir / "3-gap_and_swear_features.csv"
df = pd.read_csv(notebook3DataPath)

print("Loaded:", notebook3DataPath)
print("Rows:", len(df), "Users:", df["user_id"].nunique())
df.head(2)


Loaded: c:\Users\hus44\Code\Directed-Reading-Project\notebooks\..\data\processed\3-gap_and_swear_features.csv
Rows: 16471 Users: 869


,post_id,post_fullname,author,user_id,created_utc,created_at,title,selftext,subreddit,permalink,...,high_gap_gapmag_mean,high_gap_gapmag_max,high_gap_valence_mean_signed,high_gap_arousal_mean_signed,swear_count,swear_density,swear_present,swear_unique_count,swear_types,high_gap_swear_count
0,kt7xl7,t3_kt7xl7,Namatl,00052db50867a4da,1.610129e+09,2021-01-08 18:04:23+00:00,Streak 1 Writing Prompt,"""I was walking to the store in the evening, th...",WriteStreakEN,/r/WriteStreakEN/comments/kt7xl7/streak_1_writ...,...,NaN,NaN,NaN,NaN,0,0.0,0,0,[],0
1,ku0tf4,t3_ku0tf4,Namatl,00052db50867a4da,1.610231e+09,2021-01-09 22:18:54+00:00,Streak 2 Favorite Singer/Band,Mi favorite singer is Michael. I grew up liste...,WriteStreakEN,/r/WriteStreakEN/comments/ku0tf4/streak_2_favo...,...,NaN,NaN,NaN,NaN,0,0.0,0,0,[],0


### Choosing high gap subset

In [2]:
df_hg = df[df["high_gap_count"].fillna(0) > 0].copy()
df_non = df[df["high_gap_count"].fillna(0) == 0].copy()

print("High-gap posts:", len(df_hg))
print("Non-high-gap posts:", len(df_non))

High-gap posts: 9168
Non-high-gap posts: 7303


### Embedding model, seed sentences, and cosine similarity

We compute cosine similarities between each post and each seed, then store:

- `mh_sim_max`: max similarity across seeds
- `mh_sim_mean_top3`: mean of top 3 similarities
- `mh_best_seed_idx`, `mh_best_seed`, `mh_best_seed_sim`: interpretable "closest seed"

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

# Gathered from PH9 questionnaire
SEEDS = [
    "I have had little interest or pleasure in doing things." ,
    "Things I usually enjoy don’t feel enjoyable lately." ,
    "I’ve been feeling down, depressed, or hopeless.",
    "I’ve felt sad or empty most days.",
    "I’ve had trouble falling asleep, staying asleep, or sleeping too much.",
    "My sleep has been disrupted or unrefreshing.",
    "I’ve felt tired or like I had very little energy.",
    "Even small tasks feel exhausting.",
    "My appetite has been poor or I’ve been overeating." ,
    "My eating has changed a lot recently." ,
    "I’ve felt bad about myself, like a failure, or that I’ve let people down.",
    "I’ve been blaming myself or feeling guilty a lot.",
    "I’ve had trouble concentrating on things like reading or work.",
    "My mind feels foggy and it’s hard to focus.",
    "I’ve felt slowed down or restless and unable to sit still.",
    "Others might notice I’m moving/speaking slower, or I’ve been unusually fidgety.",
    "I’ve had thoughts that I’d be better off dead or of hurting myself.",
    "I’ve had thoughts about not wanting to be alive.",
]

texts = df_hg["text"].fillna("").astype(str).tolist()

seed_emb = model.encode(SEEDS, normalize_embeddings=True)
post_emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

sims = cosine_similarity(post_emb, seed_emb)

df_hg["mh_sim_max"] = sims.max(axis=1)
df_hg["mh_sim_mean_top3"] = np.sort(sims, axis=1)[:, -3:].mean(axis=1)

best_idx = sims.argmax(axis=1)
df_hg["mh_best_seed_idx"] = best_idx
df_hg["mh_best_seed"] = [SEEDS[i] for i in best_idx]
df_hg["mh_best_seed_sim"] = sims[np.arange(len(df_hg)), best_idx]

df_hg[["post_id","mh_sim_max","mh_best_seed_sim","mh_best_seed_idx"]].head()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8123.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 287/287 [02:59<00:00,  1.60it/s]


,post_id,mh_sim_max,mh_best_seed_sim,mh_best_seed_idx
3,j4p7n0,0.141932,0.141932,12
5,kk1866,0.354041,0.354041,1
6,18s52qq,0.230794,0.230794,0
7,18sthvq,0.255394,0.255394,13
8,18tq0cp,0.186911,0.186911,1


### Create `mh_flag`

Two options:

- `quantile`: flags the top *q* fraction of posts as mental-health-relevant 
- `fixed`: uses a fixed similarity cutoff


In [7]:
MH_MODE = "quantile"  # "quantile" or "fixed"

if MH_MODE == "quantile":
    q = 0.80
    thr = float(df_hg["mh_sim_max"].quantile(q))
    df_hg["mh_flag"] = (df_hg["mh_sim_max"] >= thr).astype(int)
    print("Using quantile threshold:", q, "->", thr)
else:
    thr = 0.45
    df_hg["mh_flag"] = (df_hg["mh_sim_max"] >= thr).astype(int)
    print("Using fixed threshold:", thr)

df_hg["mh_flag"].value_counts(dropna=False)

flagged = df_hg[df_hg["mh_flag"] == 1].copy()

seed_summary = (
    flagged.groupby(["mh_best_seed_idx","mh_best_seed"])
    .agg(
        n_posts=("post_id","count"),
        mean_best_sim=("mh_best_seed_sim","mean"),
        mean_max_sim=("mh_sim_max","mean"),
    )
    .reset_index()
    .sort_values(["n_posts","mean_best_sim"], ascending=[False, False])
)

seed_summary

cols = ["post_id","title","mh_sim_max","mh_best_seed_sim","mh_best_seed","text"]
flagged.sort_values("mh_sim_max", ascending=False)[cols].head(10)

Using quantile threshold: 0.8 -> 0.3426037669181824


,post_id,title,mh_sim_max,mh_best_seed_sim,mh_best_seed,text
8197,nhi9yo,Streak 123: My brain 🧠,0.642254,0.642254,I’ve had trouble concentrating on things like ...,"Streak 123: My brain 🧠\n\nHey folks!\n\nToday,..."
15432,1mfdgsl,Streak 12: Just Too Sleepy,0.621268,0.621268,My mind feels foggy and it’s hard to focus.,Streak 12: Just Too Sleepy\n\nSleepy… Extremel...
6983,sy2z1d,streak 130 - can’t tolerate sleep deprivation,0.601246,0.601246,My sleep has been disrupted or unrefreshing.,streak 130 - can’t tolerate sleep deprivation\...
7837,wds77n,Streak 2: I can't finish the books I start rea...,0.596121,0.596121,I’ve had trouble concentrating on things like ...,Streak 2: I can't finish the books I start rea...
15274,w4ov7e,Streak 25:,0.594804,0.594804,I’ve had trouble concentrating on things like ...,Streak 25:\n\nI'm definitely procrastinating o...
8276,p1xylr,Streak 52: intermediate plateau,0.594593,0.594593,I’ve had trouble concentrating on things like ...,"Streak 52: intermediate plateau\n\nLately, I’v..."
9729,11hz0le,Streak 169: Sleep patterns,0.582410,0.582410,My sleep has been disrupted or unrefreshing.,"Streak 169: Sleep patterns\n\nOn most days, I ..."
4157,teh9qc,Streak 1: do you have solution,0.581579,0.581579,My sleep has been disrupted or unrefreshing.,Streak 1: do you have solution\n\nI don't know...
5102,kvgv28,Streak 27: Books and reading,0.579122,0.579122,I’ve had trouble concentrating on things like ...,Streak 27: Books and reading\n\nFor the most p...
10132,1efvn6q,Streak 131: Dietary control,0.578646,0.578646,My eating has changed a lot recently.,Streak 131: Dietary control\n\nI’m trying to h...


In [8]:
DO_NON_HIGH_GAP = False

if DO_NON_HIGH_GAP:
    texts_non = df_non["text"].fillna("").astype(str).tolist()
    post_emb_non = model.encode(texts_non, normalize_embeddings=True, show_progress_bar=True)
    sims_non = cosine_similarity(post_emb_non, seed_emb)
    df_non["mh_sim_max"] = sims_non.max(axis=1)
    print("Mean mh_sim_max (high-gap):", df_hg["mh_sim_max"].mean())
    print("Mean mh_sim_max (non-high-gap):", df_non["mh_sim_max"].mean())
    
out_hg = processed_dir / "4-writestreak_posts_highgap_with_mh_embedding.csv"
df_hg.to_csv(out_hg, index=False)
print("Saved:", out_hg)

Saved: c:\Users\hus44\Code\Directed-Reading-Project\notebooks\..\data\processed\4-writestreak_posts_highgap_with_mh_embedding.csv
